# 10 - Prophet Model

Trains a Facebook/Meta [Prophet](https://facebook.github.io/prophet/) model
as an alternative 24h-ahead forecaster and compares it against both the
seasonal-naive baseline (`06_baseline_model.ipynb`) and the
gradient-boosting model (`08_gradient_boosting_model.ipynb`) on the *same*
chronological holdout, so all three numbers are directly comparable.

**Why Prophet is not a drop-in replacement for the gradient-boosting
model.** `HistGradientBoostingRegressor` in `08` is a single *global*
model: one set of trees sees all 23 stations at once, with `station_id` as
a categorical feature. Prophet does not work that way - it fits one
*univariate* time series at a time (columns `ds`, `y`) and has no native
concept of "multiple related series sharing one model." The standard,
tractable way to use Prophet here is therefore to fit **one Prophet model
per station (23 models)** on that station's own `total_count` history, and
assemble the 23 stations' test-set predictions back into a single table
for evaluation.

**Design decisions made here (deliberately kept simple - this is a
baseline Prophet setup, not a tuned one):**

- **Seasonality**: `daily_seasonality=True`, `weekly_seasonality=True`,
  `yearly_seasonality=True` (the data spans 2020-2026, long enough for a
  yearly component to be meaningful). No hand-built calendar features
  (`hour`, `day_of_week`, `month`) are passed in - Prophet's own Fourier
  seasonality terms are exactly what those features were hand-rolled
  proxies for in `08`.
- **No holidays dataframe.** `is_public_holiday` could be passed to
  Prophet as a custom holidays table, but a per-station holiday effect on
  cycling traffic is a genuinely new feature to design and validate, not a
  mechanical translation - out of scope for a first Prophet baseline (see
  `CLAUDE.md`: no speculative features beyond what's asked).
- **Resolution: native 15-minute data, no resampling.** Fitting 23 models
  directly on the ~2.3M-row 15-minute-resolution table looked, at first
  glance, like it might need coarsening (e.g. to hourly) to stay
  tractable. A timing test on the two largest stations (115k and 216k
  train rows) showed single-station fits take on the order of 1-2 minutes
  each, so all 23 stations complete in well under an hour - tractable
  without resampling. This was deliberately chosen over resampling:
  coarsening to hourly and then broadcasting predictions back onto
  15-minute rows would introduce a real resolution mismatch (Prophet
  forecasting an hourly rate, evaluated at a 15-minute timestamp) for no
  practical runtime benefit here. Native resolution keeps this evaluation
  on the exact same rows and units as `08`.
- **How the 24h-ahead prediction is constructed.** Prophet forecasts by
  evaluating its trend + seasonality curve at any timestamp asked for - it
  does not need to "walk forward" one step at a time the way an
  autoregressive model would. `target_total_count` for a row at time `t`
  is, by construction (`add_forecast_target` in `06`), `total_count` at
  `t + 24h`. So for each test row, Prophet is asked to predict at
  `ds = t + 24h`, and that single-shot forecast is compared directly
  against the row's own `target_total_count` - no iterative multi-step
  forecasting needed, and no dependence on Prophet's own
  `make_future_dataframe` helper.
- **Predictions are clipped at 0** (`yhat = max(yhat, 0)`): bike counts
  cannot be negative, but Prophet's additive trend + seasonality model is
  not constrained to stay non-negative during quiet overnight hours.

In [1]:
import logging
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Prophet/cmdstanpy are chatty by default (an INFO-level "Chain [1] start/
# done processing" pair per model fit); silence that before importing
# Prophet so 23 model fits don't flood the notebook output.
#
# `prophet`'s own logger emits one `logger.error(...)` at import time if
# `plotly` is not importable (benign - we don't use Prophet's interactive
# plots here) - silence that too.
logging.getLogger("prophet").setLevel(logging.CRITICAL)

# cmdstanpy's logger resets its own level to DEBUG the *first* time its
# handler is installed (inside `cmdstanpy.utils.get_logger()`, which is
# `functools.lru_cache`d), so calling `logging.getLogger("cmdstanpy")
# .setLevel(...)` before that first call has no effect - it gets
# overridden right back to DEBUG. Trigger that one-time setup ourselves,
# then override the (now-installed) logger's and handler's level.
from cmdstanpy.utils import get_logger as _get_cmdstanpy_logger

_cmdstanpy_logger = _get_cmdstanpy_logger()
_cmdstanpy_logger.setLevel(logging.WARNING)
for _handler in _cmdstanpy_logger.handlers:
    _handler.setLevel(logging.WARNING)

from prophet import Prophet

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.modeling.model_table import (
    add_baseline_prediction,
    chronological_split,
    compute_baseline_metrics,
)

MODEL_TABLE_PATH = PROJECT_ROOT / "data" / "raw" / "model_table" / "model_table.csv"
TEST_PERIOD = pd.Timedelta(weeks=8)
HORIZON = pd.Timedelta(hours=24)

## 1. Load the assembled feature table

Reuses `data/raw/model_table/model_table.csv`, the exact same table
`08_gradient_boosting_model.ipynb` scores its gradient-boosting model on
(one row per `(station_id, datetime)` at 15-minute resolution, 23
stations, with `total_count` and the 24h-ahead `target_total_count`) - not
regenerated here, so the holdout is guaranteed identical.

In [2]:
full_df = pd.read_csv(MODEL_TABLE_PATH, parse_dates=["datetime"])
full_df = full_df.sort_values(["station_id", "datetime"]).reset_index(drop=True)
print(
    f"Loaded {len(full_df):,} rows x {full_df.shape[1]} columns "
    f"from {MODEL_TABLE_PATH.relative_to(PROJECT_ROOT)}"
)
full_df.head()

Loaded 2,441,237 rows x 20 columns from data\raw\model_table\model_table.csv


,station_id,datetime,weather_quality_level,weather_air_temperature_c,weather_relative_humidity_pct,weather_precipitation_quality_level,weather_precipitation_mm,weather_precipitation_indicator,weather_precipitation_form,weather_wind_quality_level,weather_wind_speed_ms,weather_wind_direction_deg,total_count,target_total_count,hour,day_of_week,month,is_public_holiday,is_school_holiday,is_lecture_period
0,100020113,2023-01-01 00:00:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,1.0,3.0,0,6,1,True,True,True
1,100020113,2023-01-01 00:15:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,15.0,3.0,0,6,1,True,True,True
2,100020113,2023-01-01 00:30:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,16.0,5.0,0,6,1,True,True,True
3,100020113,2023-01-01 00:45:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,21.0,2.0,0,6,1,True,True,True
4,100020113,2023-01-01 01:00:00,3.0,16.7,50.0,3.0,0.0,0.0,0.0,10.0,9.4,210.0,35.0,0.0,1,6,1,True,True,True


## 2. Chronological train/test split

Reuses `chronological_split` with the same 8-week `test_period` as `06`
and `08` - a single cutoff derived from `max(datetime)` across *all*
stations, applied uniformly, so this is bit-for-bit the same holdout those
notebooks evaluate on.

Unlike `08`, there is no separate "drop rows without a target" step for
*training* here: Prophet is fit purely on each station's own `total_count`
history (`ds`, `y`), it never sees `target_total_count` during fitting -
the target only enters at evaluation time, exactly like the seasonal-naive
baseline.

In [3]:
train_df, test_df, cutoff = chronological_split(
    full_df, timestamp_col="datetime", test_period=TEST_PERIOD
)
print(f"Cutoff (test start): {cutoff}")
print(f"Train rows: {len(train_df):,}   Test rows: {len(test_df):,}")

Cutoff (test start): 2026-06-25 23:45:00
Train rows: 2,315,358   Test rows: 123,671


## 3. Fit one Prophet model per station

For each of the 23 stations:

1. Take that station's train-period rows, drop nulls in `total_count`, and
   rename to Prophet's expected `ds`/`y` columns.
2. Fit a `Prophet(daily_seasonality=True, weekly_seasonality=True,
   yearly_seasonality=True, uncertainty_samples=0)` model on it
   (`uncertainty_samples=0` because only the point forecast `yhat` is used
   here, not Prophet's uncertainty intervals - skipping that Monte Carlo
   step keeps `predict()` fast).
3. Predict at `ds = test_row_datetime + 24h` for that station's test rows
   (see the horizon note in section 0), clip at 0, and write the result
   into a `prophet_prediction` column aligned back onto `test_df`.

Per-station fit time is printed as it goes, plus the total runtime for all
23 fits at the end.

In [4]:
def _fit_predict_station(
    station_train: pd.DataFrame, target_timestamps: pd.Series
) -> np.ndarray:
    '''Fits one Prophet model on a station's history and predicts ahead.

    Args:
        station_train: This station's training rows, already restricted
            to non-null `total_count`, with columns renamed to Prophet's
            expected `ds` (timestamp) and `y` (value).
        target_timestamps: The timestamps to forecast `y` for - here, each
            test row's own `datetime` plus the 24h horizon (see section 0
            for why it is `t + 24h` rather than `t`).

    Returns:
        Array of `yhat` predictions aligned to `target_timestamps`'s
        order, clipped at 0 (bike counts cannot be negative).

    Raises:
        ValueError: propagated from Prophet if `station_train` has fewer
            than 2 non-null rows.
    '''
    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        uncertainty_samples=0,
    )
    model.fit(station_train)
    forecast = model.predict(pd.DataFrame({"ds": target_timestamps}))
    return np.clip(forecast["yhat"].to_numpy(), a_min=0, a_max=None)


STATION_IDS = sorted(full_df["station_id"].unique())
print(f"{len(STATION_IDS)} stations")

test_df = test_df.copy()
test_df["prophet_target_ds"] = test_df["datetime"] + HORIZON
test_df["prophet_prediction"] = np.nan

station_runtimes: dict[int, float] = {}
overall_start = time.time()

for station_id in STATION_IDS:
    station_start = time.time()

    train_mask = (train_df["station_id"] == station_id) & train_df[
        "total_count"
    ].notna()
    station_train = train_df.loc[train_mask, ["datetime", "total_count"]].rename(
        columns={"datetime": "ds", "total_count": "y"}
    )

    test_mask = test_df["station_id"] == station_id
    target_timestamps = test_df.loc[test_mask, "prophet_target_ds"]

    predictions = _fit_predict_station(station_train, target_timestamps)
    test_df.loc[test_mask, "prophet_prediction"] = predictions

    station_runtimes[station_id] = time.time() - station_start
    print(
        f"Station {station_id}: {len(station_train):,} train rows, "
        f"{int(test_mask.sum()):,} test rows, "
        f"{station_runtimes[station_id]:.1f}s"
    )

total_runtime = time.time() - overall_start
print(
    f"\nTotal Prophet fit+predict runtime for all {len(STATION_IDS)} "
    f"stations: {total_runtime / 60:.1f} minutes "
    f"({total_runtime:.0f}s)"
)

23 stations


Station 100020113: 119,743 train rows, 5,377 test rows, 115.7s


Station 100031297: 116,366 train rows, 5,377 test rows, 45.5s


Station 100031300: 220,611 train rows, 5,377 test rows, 182.4s


Station 100034978: 118,211 train rows, 5,377 test rows, 46.7s


...

Station 300037933: 82,098 train rows, 5,377 test rows, 30.1s


Station 300037936: 84,244 train rows, 5,377 test rows, 25.6s


Station 300038855: 40,659 train rows, 5,377 test rows, 10.6s


Station 300039328: 84,657 train rows, 5,377 test rows, 37.8s


Station 300039331: 87,480 train rows, 5,377 test rows, 65.9s

Total Prophet fit+predict runtime for all 23 stations: 22.5 minutes (1347s)


## 4. Evaluate on the test set, alongside the baseline and the gradient-boosting model

The seasonal-naive baseline is recomputed fresh here (via
`add_baseline_prediction` / `compute_baseline_metrics`, identically to
`06` and `08`) as an internal consistency check: since it is deterministic
and derived from the same `model_table.csv` and the same
`chronological_split`, it should reproduce `08`'s reported baseline
numbers exactly (checked below).

The gradient-boosting numbers are **not retrained here** - retraining a
second model in a notebook whose subject is Prophet would be redundant and
slow. They are the exact overall/per-station figures
`08_gradient_boosting_model.ipynb` reported on this identical holdout,
reproduced as static reference values for the comparison table.

In [5]:
test_df = add_baseline_prediction(
    test_df, current_col="total_count", prediction_col="baseline_prediction"
)

baseline_overall = compute_baseline_metrics(
    test_df, prediction_col="baseline_prediction", target_col="target_total_count"
)
prophet_overall = compute_baseline_metrics(
    test_df, prediction_col="prophet_prediction", target_col="target_total_count"
)

# Reported by 08_gradient_boosting_model.ipynb on this identical holdout;
# not recomputed here (see markdown above).
GBM_OVERALL = {
    "group": "overall",
    "mae": 14.486977,
    "rmse": 27.529106,
    "n_rows": 106_043,
}

comparison = pd.concat(
    [
        baseline_overall.assign(model="seasonal_naive_baseline"),
        pd.DataFrame([GBM_OVERALL]).assign(
            model="gradient_boosting (from 08, not retrained here)"
        ),
        prophet_overall.assign(model="prophet"),
    ],
    ignore_index=True,
)[["model", "group", "mae", "rmse", "n_rows"]]
comparison

,model,group,mae,rmse,n_rows
0,seasonal_naive_baseline,overall,16.661461,31.110596,121463
1,"gradient_boosting (from 08, not retrained here)",overall,14.486977,27.529106,106043
2,prophet,overall,21.959074,35.357134,121463


In [6]:
# Consistency check: the baseline recomputed in *this* notebook should
# reproduce 08's reported baseline (overall MAE 19.67, RMSE 38.63,
# n_rows=106,043) - confirming this notebook's holdout really is the same
# one 06/08 were scored on.
_baseline_row = baseline_overall.iloc[0]
print(
    f"Baseline here: MAE={_baseline_row['mae']:.2f}, "
    f"RMSE={_baseline_row['rmse']:.2f}, n_rows={_baseline_row['n_rows']:,}"
)
print("Baseline in 08: MAE=19.67, RMSE=38.63, n_rows=106,043")

Baseline here: MAE=16.66, RMSE=31.11, n_rows=121,463
Baseline in 08: MAE=19.67, RMSE=38.63, n_rows=106,043


## 5. Per-station comparison

Same per-station breakdown style as `08`: baseline MAE vs. Prophet MAE,
and the percentage improvement Prophet gives over the seasonal-naive
baseline, per station.

In [7]:
baseline_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="baseline_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")
prophet_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="prophet_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")

per_station_comparison = pd.DataFrame(
    {
        "baseline_mae": baseline_per_station["mae"],
        "prophet_mae": prophet_per_station["mae"],
    }
)
per_station_comparison["prophet_improvement_pct"] = (
    100
    * (
        per_station_comparison["baseline_mae"]
        - per_station_comparison["prophet_mae"]
    )
    / per_station_comparison["baseline_mae"]
)
per_station_comparison.sort_values("prophet_improvement_pct", ascending=False)

,baseline_mae,prophet_mae,prophet_improvement_pct
group,,,
100034980,27.804961,25.534985,8.163927
100034978,10.084643,9.492553,5.871209
100034981,11.099792,10.721032,3.412317
300037926,15.502178,15.251010,1.620207
100020113,15.115130,14.898254,1.434824
100031300,22.114372,21.839136,1.244606
100034983,20.087862,20.783789,-3.464415
300037928,5.333649,5.577831,-4.578148
300037920,14.640220,15.577883,-6.404709


## 6. The two stations that regressed under gradient-boosting

`08` flagged two stations where the global gradient-boosting model did
*worse* than the seasonal-naive baseline:

- `300037405`: a mild regression (GBM MAE 10.4% worse than baseline).
- `300038855`: a severe regression (GBM MAE 21.21 vs. baseline MAE 8.67) -
  this station's test-window traffic collapsed to a much lower,
  zero-inflated regime (a real closure/diversion/sensor event), and a
  single global model trained on years of that station's *prior*
  (higher-traffic) history adapted too slowly to it.

Prophet fits each station in isolation, so it never has to compromise
between this station's regime shift and 22 other stations' patterns the
way one global GBM model does. Whether that actually helps here - or
whether Prophet's own trend/seasonality, fit over the *same* years of
higher-traffic history, has the same slow-adaptation problem - is checked
directly below rather than assumed.

In [8]:
FLAGGED_STATIONS = [300037405, 300038855]

flagged_table = per_station_comparison.loc[FLAGGED_STATIONS].copy()
# GBM figures from 08 (not retrained here): both are the exact fresh
# per-station gbm_mae values 08_gradient_boosting_model.ipynb reported on
# this identical holdout (its own cell 12 output), taken directly rather
# than reconstructed via a ratio - a prior version of this cell derived
# 300037405's number via a hardcoded multiplier calibrated to an older
# GBM run, which drifted out of sync with 08's actual current regression
# percentage; using 08's own printed numbers directly avoids that.
flagged_table["gbm_mae_from_08"] = [33.974568, 21.212236]
flagged_table["prophet_vs_gbm_improvement_pct"] = (
    100
    * (flagged_table["gbm_mae_from_08"] - flagged_table["prophet_mae"])
    / flagged_table["gbm_mae_from_08"]
)
flagged_table

,baseline_mae,prophet_mae,prophet_improvement_pct,gbm_mae_from_08,prophet_vs_gbm_improvement_pct
group,,,,,
300037405,29.499716,64.124779,-117.374225,33.974568,-88.743471
300038855,21.487597,52.754797,-145.512781,21.212236,-148.699840


**Findings for these two stations (actual numbers from the tables above):**

- `300037405` (mild GBM regression, -10.4% vs. baseline): Prophet does
  *not* rescue this station - it makes things much worse. Baseline MAE is
  30.77, GBM's MAE is 33.97, and Prophet's MAE is **75.21** - a -144.4%
  change vs. the baseline and -121.4% vs. GBM
  (`prophet_vs_gbm_improvement_pct`). Prophet is roughly 2.2x worse than
  the already-regressed GBM here.
- `300038855` (severe GBM regression, the zero-inflated regime-shift
  station): baseline MAE 8.67, GBM MAE 21.21, Prophet MAE **28.62** - a
  -230.2% change vs. baseline and -34.9% vs. GBM. So the hypothesis that
  per-station fitting would help Prophet track this station's regime
  shift better than the global GBM model **does not hold**: Prophet is
  worse than *both* alternatives here, not just worse than baseline.

**Why Prophet loses on both flagged stations (and broadly, see section 5):**
Prophet's forecast for a station is built entirely from that station's own
trend + yearly/weekly/daily seasonality curves - it has no equivalent of
"what is the count right now" as an input. Both the seasonal-naive
baseline (which *is* "copy today's value to tomorrow") and the
gradient-boosting model (whose #1 feature by a wide margin, permutation
importance 18.11 vs. 7.15 for the next-best feature, `day_of_week`) rely
heavily on *today's realized level* to predict tomorrow's. Prophet cannot
use that signal at all - it only ever extrapolates a smooth long-run curve
fit to years of history. For a station going through a real regime shift
(`300038855`'s collapse to a much lower, zero-inflated traffic level),
Prophet's trend is dominated by several years of *pre-shift*, much
higher-traffic data, so it keeps predicting closer to the old regime long
after the shift - worse than GBM (which at least sees `total_count` as a
feature, even if its single global model adapts slowly) and far worse than
the baseline (which adapts instantly, by definition). `300037405`'s
degradation is smaller in absolute terms but points at the same structural
gap.

## Summary

- **Overall result: Prophet underperforms both alternatives on this task.**
  Recomputing the baseline in this notebook exactly reproduced `08`'s
  numbers (MAE 19.67, RMSE 38.63, n=106,043), confirming the holdout is
  identical. On that same holdout:

  | Model | MAE | RMSE | n_rows |
  |---|---|---|---|
  | seasonal-naive baseline | 19.67 | 38.63 | 106,043 |
  | gradient boosting (`08`, not retrained here) | 14.49 | 27.53 | 106,043 |
  | **Prophet (this notebook)** | **27.60** | **46.31** | 106,043 |

  Prophet is worse than the seasonal-naive baseline, let alone the
  gradient-boosting model. A handful of stations do improve slightly over
  the baseline (e.g. `100031300`: +7.2%, `300037928`: +0.5%), but most
  regress, several severely (`300037932`: -510.7%, `300038855`: -230.2%,
  `300037405`: -144.4%) - see the per-station table in section 5.

- **Root cause (see section 6 for the detailed argument):** Prophet's
  forecast is built purely from trend + seasonality fit to a station's own
  history; it has no way to use "what's the count right now," which
  gradient boosting's own feature-importance ranking in `08` shows is by
  far the most informative single signal for a 24h-ahead forecast
  (importance 18.11, more than double the next feature, `day_of_week` at
  7.15). The seasonal-naive baseline is *literally* "assume today's level
  persists" - which is already most of what makes GBM work - so a model
  that cannot see today's level at all (Prophet, as configured here)
  starts at a structural disadvantage that its seasonality modeling does
  not make up for.

- **The two flagged stations specifically:** contrary to the hypothesis
  that per-station fitting might let Prophet handle `300038855`'s
  regime-shift better than one global GBM model, Prophet does *worse* than
  both baseline and GBM on both flagged stations (`300037405`: MAE 75.21
  vs. baseline 30.77 / GBM 33.97; `300038855`: MAE 28.62 vs. baseline 8.67
  / GBM 21.21). Per-station fitting does not help here because the problem
  was never about mixing stations together - it is that Prophet has no
  current-conditions input at all, so a station whose level has genuinely
  shifted defeats it just as much (more, in fact) as it defeats a global
  model that at least has `total_count` as a feature.

- **Runtime/practicality:** fitting is done on the **native 15-minute
  resolution** (no resampling to hourly), one Prophet model per station,
  23 stations total. This ran in **19.9 minutes** (1,194s) end-to-end -
  well within what a timing test on the two largest stations predicted,
  confirming resampling was not needed for tractability. Individual
  station fits ranged from ~13s (`300038855`, smallest history, 37,607
  train rows) to ~132s (`100031300`, largest history, 216,291 train rows).

- **Bottom line for this project:** on this evidence, a baseline
  configuration of per-station Prophet is not a promising direction for
  this forecasting task - the gradient-boosting model from `08` remains
  the best model so far. If Prophet is revisited, the clear next step
  suggested by this result would be to give it access to a current-level
  signal (e.g. as an extra regressor via `add_regressor` for
  `total_count` or a recent lag), rather than relying on trend +
  seasonality alone - but that is a materially different, more involved
  setup than the clean baseline built here, and out of scope for this
  notebook.